# 🦴 Treino Fase 3 - Classificação de Tumores Ósseos
## Otimizado para Google Colab (T4/V100/A100)

**Configurações:**
- ⚡ Leitura do SSD local (10-20x mais rápido que Drive)
- 🎯 Batch size otimizado automaticamente por GPU
- 🔄 Checkpoint automático com retomada
- 🚀 Mixed Precision (AMP) habilitado
- 💾 Salva resultados no Drive (não perde se desconectar)
- 🔧 CLAHE pré-processado + Workers otimizados

## 📦 1. Instalação de Dependências

In [ ]:
!pip install -q timm rich scikit-learn
print("✅ Dependências instaladas!")

## 🔗 2. Montagem do Google Drive

In [ ]:
from google.colab import drive
import os

# Monta o Google Drive
drive.mount('/content/drive')

# 🔧 AJUSTE ESTE CAMINHO PARA O SEU DRIVE
DRIVE_BASE = '/content/drive/MyDrive/TCC2'
ZIP_PATH = os.path.join(DRIVE_BASE, 'Fase3.zip')  # Arquivo ZIP da pasta Fase3

# Verifica se o ZIP existe
if os.path.exists(ZIP_PATH):
    print(f"✅ ZIP encontrado: {ZIP_PATH}")
else:
    print(f"❌ ERRO: ZIP não encontrado em {ZIP_PATH}")
    print("Por favor, ajuste DRIVE_BASE ou certifique-se que Fase3.zip está no Drive.")
    raise FileNotFoundError(f"ZIP não encontrado: {ZIP_PATH}")

## ⚡ 3. Descompactar Dados no SSD Local (CRÍTICO PARA PERFORMANCE!)

In [ ]:
import zipfile
from pathlib import Path

# Diretório local (SSD rápido do Colab)
LOCAL_DIR = '/content/Fase3'

# Descompacta dados se ainda não existir
if not os.path.exists(LOCAL_DIR):
    print(f"📦 Extraindo dataset para SSD local...")
    print(f"⚠️  Isso pode levar 2-5 minutos dependendo do tamanho")
    
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        # Mostra progresso
        file_list = zip_ref.namelist()
        total = len(file_list)
        
        for i, file in enumerate(file_list):
            zip_ref.extract(file, '/content')
            if (i + 1) % 100 == 0 or (i + 1) == total:
                print(f"Progresso: {i+1}/{total} arquivos ({((i+1)/total*100):.1f}%)")
    
    print("✅ Dataset extraído com sucesso no SSD local!")
    print(f"📂 Localização: {LOCAL_DIR}")
else:
    print(f"✅ Dataset já existe no SSD local: {LOCAL_DIR}")
    print("⏭️  Pulando extração...")

# Verifica estrutura
print("\n📁 Estrutura de diretórios:")
for item in os.listdir(LOCAL_DIR):
    item_path = os.path.join(LOCAL_DIR, item)
    if os.path.isdir(item_path):
        num_files = sum(len(files) for _, _, files in os.walk(item_path))
        print(f"  ├── {item}/ ({num_files} arquivos)")
    else:
        print(f"  ├── {item}")

## 🔍 4. Detecção Automática de GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"🎮 GPU Detectada: {gpu_name}")
    print(f"💾 VRAM Total: {gpu_memory:.1f} GB")
    
    # Configurações otimizadas por GPU
    if 'T4' in gpu_name:
        GPU_TYPE = 'T4'
        BATCH_MULTIPLIER = 1.0
        print("⚙️  T4 (15GB) - Batch conservador para estabilidade")
    elif 'V100' in gpu_name:
        GPU_TYPE = 'V100'
        BATCH_MULTIPLIER = 1.3
        print("⚙️  V100 (16GB) - Batch +30%")
    elif 'A100' in gpu_name:
        GPU_TYPE = 'A100'
        BATCH_MULTIPLIER = 3.0
        print("⚙️  A100 (40GB) - Batch +200% 🚀")
    else:
        GPU_TYPE = 'OTHER'
        BATCH_MULTIPLIER = 1.0
        print(f"⚙️  {gpu_name} - Batch padrão")
else:
    print("❌ GPU não detectada!")
    print("Vá em Runtime > Change runtime type > Hardware accelerator > GPU")
    raise RuntimeError("GPU necessária")

## 📚 5. Imports

In [ ]:
import pandas as pd
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image
import timm
import numpy as np
import random
import time
import gc
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.progress import Progress, SpinnerColumn, BarColumn, TextColumn, TimeRemainingColumn
from rich.live import Live
from rich import box

console = Console()
print("✅ Imports carregados!")

## ⚙️ 6. Configurações

In [ ]:
# ================= CONFIGURAÇÕES FASE 3 =================

# 🔥 DADOS: Leitura do SSD LOCAL (rápido)
DIR_CNN = os.path.join(LOCAL_DIR, 'classificacao_crops_cnn')
DIR_VIT = os.path.join(LOCAL_DIR, 'classificacao_crops_vit')
DIR_TESTE = os.path.join(LOCAL_DIR, 'classificacao_crops_teste')
CSV_CNN = os.path.join(DIR_CNN, 'dataset_kfold_controle.csv')
CSV_VIT = os.path.join(DIR_VIT, 'dataset_kfold_controle_vit.csv')

# 💾 SAÍDAS: Salvam no DRIVE (não perde se desconectar)
DRIVE_OUTPUT = os.path.join(DRIVE_BASE, 'resultados_fase3')
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
PASTA_PESOS_RAD = os.path.join(LOCAL_DIR, 'pesos_radimagenet')
PASTA_SAIDA_MODELOS = os.path.join(DRIVE_OUTPUT, 'modelos')  # ← DRIVE
ARQUIVO_LOG = os.path.join(DRIVE_OUTPUT, 'RESULTADOS_FASE3.csv')  # ← DRIVE

DEVICE = 'cuda'
NUM_WORKERS = 4  # Linux no Colab = fork eficiente
SEED = 42

# Hiperparâmetros
EPOCHS = 60
PATIENCE = 12
LR = 1e-4
IMG_SIZE = 384
LABEL_SMOOTHING = 0.1
DROP_PATH_RATE = 0.2

# 🎯 Batch sizes ajustados por GPU (conservadores para T4)
MODELOS_PARA_TREINO = [
    {'name': 'resnet50', 'batch': int(16 * BATCH_MULTIPLIER), 'tipo': 'cnn', 'radimagenet': True},
    {'name': 'densenet121', 'batch': int(12 * BATCH_MULTIPLIER), 'tipo': 'cnn', 'radimagenet': True},
    {'name': 'tf_efficientnetv2_l', 'batch': int(4 * BATCH_MULTIPLIER), 'tipo': 'cnn', 'radimagenet': False},
    {'name': 'swinv2_base_window12to24_192to384_22kft1k', 'batch': int(5 * BATCH_MULTIPLIER), 'tipo': 'vit', 'radimagenet': False},  # Batch 5 para T4 (seguro)
    {'name': 'beitv2_base_patch16_224.in1k_ft_in22k', 'batch': int(8 * BATCH_MULTIPLIER), 'tipo': 'vit', 'radimagenet': False},
]

console.print("[green]✅ Configurações definidas![/]")
console.print(f"[cyan]📊 Modelos: {len(MODELOS_PARA_TREINO)}[/]")
console.print(f"[yellow]🎮 GPU: {GPU_TYPE} | Batch Multiplier: {BATCH_MULTIPLIER}x[/]")
console.print(f"[magenta]📂 Dados (SSD): {LOCAL_DIR}[/]")
console.print(f"[magenta]💾 Saída (Drive): {DRIVE_OUTPUT}[/]")

## 🔧 7. Funções Auxiliares

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

class BoneDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df
        self.root_dir = root_dir
        self.transform = transform
        self.class_map = {'benigno': 0, 'maligno': 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.root_dir, row['split_group'], row['class_name'], row['image_id'])
        
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE))
            
        label = self.class_map[row['class_name']]
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

def carregar_pesos_radimagenet(model, model_name):
    nome_arquivo = ''
    if 'resnet50' in model_name:
        nome_arquivo = 'RadImageNet-ResNet50_notop.pth'
    elif 'densenet121' in model_name:
        nome_arquivo = 'DenseNet121.pt'
    
    caminho = os.path.join(PASTA_PESOS_RAD, nome_arquivo)
    if not os.path.exists(caminho):
        caminho_alt = os.path.join(PASTA_PESOS_RAD, f"{model_name}.pt")
        if os.path.exists(caminho_alt): caminho = caminho_alt
    
    if os.path.exists(caminho):
        console.print(f"[cyan] -> 🩻 RadImageNet: {os.path.basename(caminho)}[/]")
        try:
            state_dict = torch.load(caminho, map_location='cpu')
            model.load_state_dict(state_dict, strict=False)
            console.print(f"[bold green] -> ✓ Pesos Médicos Carregados![/]")
            return True
        except Exception as e:
            console.print(f"[bold red] -> ✗ Erro: {e}[/]")
    else:
        console.print(f"[yellow] -> ⚠ RadImageNet não encontrado. Usando ImageNet.[/]")
    return False

def criar_tabela_dashboard():
    table = Table(title="Histórico de Treinamento", box=box.ROUNDED)
    table.add_column("Ep", justify="center", style="cyan", no_wrap=True)
    table.add_column("Train Loss", justify="right", style="magenta")
    table.add_column("Val Loss", justify="right", style="magenta")
    table.add_column("Val F1", justify="right", style="green")
    table.add_column("Status", justify="center")
    return table

console.print("[green]✅ Funções definidas![/]")

## 🏋️ 8. Função de Treino

In [ ]:
def treinar_ciclo(config):
    model_name = config['name']
    batch_size = config['batch']
    tipo_modelo = config['tipo']
    usar_rad = config['radimagenet']
    
    if tipo_modelo == 'cnn':
        pasta_treino = DIR_CNN
        arquivo_csv = CSV_CNN
        info_dados = "Original"
    else:
        pasta_treino = DIR_VIT
        arquivo_csv = CSV_VIT
        info_dados = "Expandido (10x)"

    current_size = 224 if 'beit' in model_name else IMG_SIZE

    console.print(Panel.fit(
        f"[bold white]Modelo:[/][bold yellow] {model_name}[/]\n"
        f"[bold white]Tipo:[/][cyan] {tipo_modelo.upper()}[/] | [bold white]Dados:[/][cyan] {info_dados}[/]\n"
        f"[bold white]Batch:[/][green] {batch_size}[/] | [bold white]Size:[/][green] {current_size}px[/]",
        title="🚀 Configuração", border_style="blue"
    ))

    with console.status("[bold green]Preparando pipeline...", spinner="dots"):
        
        train_transforms = transforms.Compose([
            transforms.RandomResizedCrop(current_size, scale=(0.85, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        
        val_transforms = transforms.Compose([
            transforms.Resize((current_size, current_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

        df = pd.read_csv(arquivo_csv)
        df_train = df[(df['split_group'] == 'treino_kfold') & (df['fold'] != 0)]
        df_val = df[(df['split_group'] == 'treino_kfold') & (df['fold'] == 0)]
        
        test_images = []
        for cls_name in ['benigno', 'maligno']:
            path_cls = os.path.join(DIR_TESTE, cls_name)
            if os.path.exists(path_cls):
                for f in os.listdir(path_cls):
                    if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                        test_images.append({'split_group': '', 'class_name': cls_name, 'image_id': f})
        df_test = pd.DataFrame(test_images)

        if len(df_test) == 0:
            raise ValueError(f"❌ Teste vazio: {DIR_TESTE}")

        targets = df_train['class_name'].map({'benigno': 0, 'maligno': 1}).values
        class_counts = np.bincount(targets)
        weights = 1. / class_counts
        samples_weights = weights[targets]
        sampler = WeightedRandomSampler(samples_weights, len(samples_weights), replacement=True)

        loader_kwargs = {
            'num_workers': NUM_WORKERS,
            'pin_memory': True,
            'persistent_workers': True if NUM_WORKERS > 0 else False
        }

        train_loader = DataLoader(BoneDataset(df_train, pasta_treino, train_transforms), batch_size=batch_size, sampler=sampler, **loader_kwargs)
        val_loader = DataLoader(BoneDataset(df_val, pasta_treino, val_transforms), batch_size=batch_size, shuffle=False, **loader_kwargs)
        test_loader = DataLoader(BoneDataset(df_test, DIR_TESTE, val_transforms), batch_size=batch_size, shuffle=False, **loader_kwargs)

        try:
            model = timm.create_model(model_name, pretrained=True, num_classes=2, drop_path_rate=DROP_PATH_RATE)
        except TypeError:
            console.print(f"[yellow]⚠ {model_name} sem drop_path_rate[/]")
            model = timm.create_model(model_name, pretrained=True, num_classes=2)
        
        if usar_rad:
            carregar_pesos_radimagenet(model, model_name)
        
        model = model.to(DEVICE)
        lr_atual = LR / 2 if tipo_modelo == 'vit' else LR
        criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
        optimizer = optim.AdamW(model.parameters(), lr=lr_atual, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
        scaler = torch.amp.GradScaler('cuda')  # AMP sempre ativo

    best_loss = float('inf')
    patience_count = 0
    save_path = os.path.join(PASTA_SAIDA_MODELOS, f"{model_name}_fase3.pth")
    checkpoint_path = os.path.join(PASTA_SAIDA_MODELOS, f"{model_name}_checkpoint.pth")
    os.makedirs(PASTA_SAIDA_MODELOS, exist_ok=True)
    start_time = time.time()
    start_epoch = 0

    if os.path.exists(checkpoint_path):
        console.print(f"[yellow]📥 Checkpoint encontrado![/]")
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
        model.load_state_dict(checkpoint['model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        scheduler.load_state_dict(checkpoint['scheduler_state'])
        best_loss = checkpoint['best_loss']
        start_epoch = checkpoint['epoch'] + 1
        patience_count = checkpoint['patience_count']
        console.print(f"[green]✓ Retomando: Ep {start_epoch}/{EPOCHS} | Loss: {best_loss:.4f}[/]")
    
    dashboard_table = criar_tabela_dashboard()
    progress = Progress(SpinnerColumn(), TextColumn("{task.description}"), BarColumn(), TextColumn("{task.percentage:>3.0f}%"), TimeRemainingColumn())
    
    with Live(Panel(dashboard_table, title="Monitoramento", border_style="green"), refresh_per_second=4) as live:
        for epoch in range(start_epoch, EPOCHS):
            model.train()
            train_loss = 0
            batches_ok = 0
            task_id = progress.add_task(f"[cyan]Ep {epoch+1}", total=len(train_loader))
            
            for i, (imgs, lbls) in enumerate(train_loader):
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                optimizer.zero_grad()
                
                with torch.amp.autocast('cuda'):
                    outputs = model(imgs)
                    loss = criterion(outputs, lbls)
                
                if torch.isnan(loss) or torch.isinf(loss):
                    console.print(f"[bold red]⚠ NaN batch {i}[/]")
                    continue

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                train_loss += loss.item()
                batches_ok += 1

                if i % 10 == 0:
                    live.update(Panel(dashboard_table, title=f"Ep {epoch+1} | Batch {i}/{len(train_loader)}", border_style="green"))

            progress.remove_task(task_id)

            if batches_ok == 0:
                console.print(f"[bold red]❌ Ep {epoch+1} sem batches válidos![/]")
                break

            avg_train_loss = train_loss / batches_ok
            scheduler.step(epoch + avg_train_loss)

            model.eval()
            val_loss = 0
            all_preds, all_lbls = [], []
            with torch.no_grad():
                for imgs, lbls in val_loader:
                    imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                    outputs = model(imgs)
                    val_loss += criterion(outputs, lbls).item()
                    _, preds = torch.max(outputs, 1)
                    all_preds.extend(preds.cpu().numpy())
                    all_lbls.extend(lbls.cpu().numpy())

            val_loss /= len(val_loader)
            val_f1 = f1_score(all_lbls, all_preds, average='macro')
            
            if val_loss < best_loss:
                best_loss = val_loss
                patience_count = 0
                torch.save(model.state_dict(), save_path)
                status_icon = "[bold green]★ SALVO[/]"
            else:
                patience_count += 1
                status_icon = f"[yellow]{patience_count}/{PATIENCE}[/]"

            torch.save({
                'epoch': epoch,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'scheduler_state': scheduler.state_dict(),
                'best_loss': best_loss,
                'patience_count': patience_count,
            }, checkpoint_path)

            dashboard_table.add_row(f"{epoch+1}", f"{avg_train_loss:.4f}", f"{val_loss:.4f}", f"{val_f1:.4f}", status_icon)
            live.update(Panel(dashboard_table, title="Monitoramento", border_style="green"))

            if patience_count >= PATIENCE:
                dashboard_table.add_row("STOP", "-", "-", "-", "[bold red]EARLY STOP[/]")
                if os.path.exists(checkpoint_path):
                    os.remove(checkpoint_path)
                break

    console.print("\n[bold cyan]--- TESTE FINAL ---[/]")
    model.load_state_dict(torch.load(save_path))
    model.eval()
    test_preds, test_targets, test_probs = [], [], []
    with torch.no_grad():
        for imgs, lbls in test_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            outputs = model(imgs)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            _, preds = torch.max(outputs, 1)
            test_preds.extend(preds.cpu().numpy())
            test_targets.extend(lbls.cpu().numpy())
            test_probs.extend(probs.cpu().numpy())

    if len(test_targets) == 0:
        console.print("[bold red]❌ Test vazio![/]")
        return None

    return {
        'Modelo': model_name,
        'Tipo': tipo_modelo.upper(),
        'RadImageNet': 'Sim' if usar_rad else 'Nao',
        'Resolucao': f"{current_size}x{current_size}",
        'Val_Loss': round(best_loss, 4),
        'Acuracia': round(accuracy_score(test_targets, test_preds), 4),
        'F1_Score': round(f1_score(test_targets, test_preds), 4),
        'Recall': round(recall_score(test_targets, test_preds), 4),
        'AUC': round(roc_auc_score(test_targets, test_probs), 4) if len(set(test_targets)) > 1 else 0,
        'Tempo_min': round((time.time() - start_time)/60, 2)
    }

console.print("[green]✅ Função de treino definida![/]")

## 🚀 9. EXECUTAR TREINAMENTO

In [ ]:
def main():
    console.rule("[bold blue]FASE 3: TREINAMENTO COMPLETO[/]")
    resultados = []
    
    if os.path.exists(ARQUIVO_LOG):
        resultados = pd.read_csv(ARQUIVO_LOG).to_dict('records')

    for config in MODELOS_PARA_TREINO:
        nome_mod = config['name']
        if any(r['Modelo'] == nome_mod for r in resultados):
            console.print(f"[yellow]⏭️  PULANDO {nome_mod} (já treinado)[/]")
            continue
            
        gc.collect()
        torch.cuda.empty_cache()
        
        try:
            res = treinar_ciclo(config)
            if res:
                resultados.append(res)
                pd.DataFrame(resultados).to_csv(ARQUIVO_LOG, index=False)
                console.print(Panel(
                    f"Recall: [bold green]{res['Recall']}[/] | AUC: [bold magenta]{res['AUC']}[/]",
                    title=f"✅ {nome_mod}"
                ))
        except Exception as e:
            console.print_exception()
            console.print(f"[bold red]❌ Erro em {nome_mod}[/]")

    console.rule("[bold green]🎉 CONCLUÍDO![/]")
    
    if os.path.exists(ARQUIVO_LOG):
        df_results = pd.read_csv(ARQUIVO_LOG)
        console.print("\n[bold cyan]📊 RESULTADOS:[/]")
        console.print(df_results.to_string(index=False))

main()

## 📊 10. Visualizar Resultados

In [ ]:
import matplotlib.pyplot as plt

df_results = pd.read_csv(ARQUIVO_LOG)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Comparação de Modelos - Fase 3', fontsize=16, fontweight='bold')

axes[0, 0].barh(df_results['Modelo'], df_results['Acuracia'], color='skyblue')
axes[0, 0].set_xlabel('Acurácia')
axes[0, 0].set_title('Acurácia')
axes[0, 0].set_xlim(0, 1)

axes[0, 1].barh(df_results['Modelo'], df_results['F1_Score'], color='lightcoral')
axes[0, 1].set_xlabel('F1-Score')
axes[0, 1].set_title('F1-Score')
axes[0, 1].set_xlim(0, 1)

axes[1, 0].barh(df_results['Modelo'], df_results['AUC'], color='lightgreen')
axes[1, 0].set_xlabel('AUC')
axes[1, 0].set_title('AUC-ROC')
axes[1, 0].set_xlim(0, 1)

axes[1, 1].barh(df_results['Modelo'], df_results['Tempo_min'], color='plum')
axes[1, 1].set_xlabel('Tempo (min)')
axes[1, 1].set_title('Tempo de Treino')

plt.tight_layout()
plt.savefig(os.path.join(DRIVE_OUTPUT, 'graficos.png'), dpi=300, bbox_inches='tight')
plt.show()

## 💾 11. Download dos Resultados

In [ ]:
from google.colab import files
import zipfile

zip_path = '/content/resultados_fase3.zip'
with zipfile.ZipFile(zip_path, 'w') as zipf:
    zipf.write(ARQUIVO_LOG, 'RESULTADOS_FASE3.csv')
    
    grafico = os.path.join(DRIVE_OUTPUT, 'graficos.png')
    if os.path.exists(grafico):
        zipf.write(grafico, 'graficos.png')
    
    for arq in os.listdir(PASTA_SAIDA_MODELOS):
        if arq.endswith('_fase3.pth'):
            zipf.write(os.path.join(PASTA_SAIDA_MODELOS, arq), f'modelos/{arq}')

print("✅ ZIP criado!")
files.download(zip_path)